# 📥 Download Historical Baseline Data (1981–2023)

## One-Time Process — Estimated Duration: ~3–4 hours

This notebook downloads **43 years of daily climate data** from Google Earth Engine
for all 64 Bangladesh districts and computes day-of-year climatology statistics.

### What gets downloaded:
| Dataset | Period | Resolution | Size (approx) |
|---------|--------|------------|---------------|
| CHIRPS Daily Rainfall | 1981–2023 | 5.5km/district | ~1M rows |
| ERA5-Land Temperature | 1981–2023 | 11km/district | ~1M rows |

### What gets computed:
- **Day-of-year climatology** (mean, std, percentiles p10/p25/p50/p75/p90)
- This baseline is used to compute anomalies in the monitoring pipeline

### Key features:
- ✅ **Checkpoint/resume**: Each year is saved as a CSV — restart safely after interruptions
- ✅ **Progress tracking**: Real-time ETA estimates
- ✅ **Rate limiting**: Respects GEE quotas with configurable delays

### After completion:
1. Download the climatology CSV files from `/content/historical/`
2. Include them in your `production_pipeline/data/historical/` directory
3. They will be auto-loaded by the main testing notebook via `bundled` mode

---
**⚠️ Prerequisites:** You need a Google Earth Engine account with a Cloud Project.


In [ ]:
# STEP 1: Install Dependencies
print('='*80)
print('STEP 1 — Installing dependencies')
print('='*80)

import sys, subprocess

PACKAGES = [
    'earthengine-api', 'geemap', 'geopandas', 'shapely',
    'pyproj', 'fiona', 'pandas', 'numpy', 'pyarrow',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
print('✅ Dependencies installed')


In [ ]:
# STEP 2: Setup Google Earth Engine
print('='*80)
print('STEP 2 — Authenticate & initialize GEE')
print('='*80)

import os, sys

# Ensure production_pipeline is importable
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# ╔══════════════════════════════════════════════════════════════╗
# ║  IMPORTANT: Set YOUR GEE Cloud Project name below          ║
# ╚══════════════════════════════════════════════════════════════╝
PROJECT_NAME = os.environ.get('GEE_PROJECT', 'genai-bangladesh-drought-demo')

from production_pipeline.setup_colab import setup_gee_for_colab
setup_gee_for_colab(PROJECT_NAME)
print(f'✅ GEE ready with project: {PROJECT_NAME}')


In [ ]:
# STEP 3: Import modules
print('='*80)
print('STEP 3 — Import pipeline modules')
print('='*80)

import time
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import geopandas as gpd
    from shapely import wkt
except ImportError:
    raise RuntimeError('geopandas/shapely required. Re-run Step 1.')

from production_pipeline.extractors.static_extractor import load_hdx_boundaries
from production_pipeline.extractors.historical_extractor import (
    extract_chirps_historical,
    extract_era5_historical,
    compute_chirps_climatology,
    compute_era5_climatology,
    save_climatology_bundle,
    get_latest_chirps_date,
    get_latest_era5_date,
)

# Output directories
BASE_DIR = Path('/content')
HIST_DIR = BASE_DIR / 'historical'
RAW_DIR  = HIST_DIR / 'raw'
HIST_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('✅ All imports successful')
print(f'📁 Output directory: {HIST_DIR}')


In [ ]:
# STEP 4: Load Bangladesh district boundaries
print('='*80)
print('STEP 4 — Load Bangladesh district boundaries')
print('='*80)

print('\n📍 Loading district boundaries from bundled static data...')

hdx_df = load_hdx_boundaries()
print(f'   Raw HDX data: {len(hdx_df)} rows')

# Build GeoDataFrame with proper geometry
if hasattr(hdx_df, 'geometry') and hdx_df.geometry is not None:
    # Check for canonical columns first, fall back to plain names
    id_col = 'district_id_canonical' if 'district_id_canonical' in hdx_df.columns else 'district_id'
    name_col = 'district_name_canonical' if 'district_name_canonical' in hdx_df.columns else 'district_name'
    districts_gdf = hdx_df[[id_col, name_col, 'geometry']].copy()
    districts_gdf = districts_gdf.rename(columns={id_col: 'district_id', name_col: 'district_name'})
elif 'geometry_wkt' in hdx_df.columns:
    hdx_df['geometry'] = hdx_df['geometry_wkt'].apply(wkt.loads)
    id_col = 'district_id_canonical' if 'district_id_canonical' in hdx_df.columns else 'district_id'
    name_col = 'district_name_canonical' if 'district_name_canonical' in hdx_df.columns else 'district_name'
    districts_gdf = gpd.GeoDataFrame(
        hdx_df[[id_col, name_col, 'geometry']].rename(
            columns={id_col: 'district_id', name_col: 'district_name'}
        ),
        geometry='geometry', crs='EPSG:4326'
    )
else:
    raise RuntimeError('No geometry found in HDX data')

print(f'\n✅ District GeoDataFrame built: {len(districts_gdf)} districts')
print(f'   CRS: {districts_gdf.crs}')
print(f'\n📋 Sample districts:')
for i, row in districts_gdf.head(10).iterrows():
    print(f'   {row["district_id"]:>8}  {row["district_name"]}')


In [ ]:
# ========================================
# CRITICAL VERIFICATION: Real Districts
# ========================================

print('='*80)
print('🔍 VERIFYING DISTRICT DATA')
print('='*80)

# Check district count
num_districts = len(districts_gdf)
print(f'\n📊 District count: {num_districts}')

if num_districts != 64:
    print('\n❌ CRITICAL ERROR: Expected 64 districts!')
    print(f'   Got: {num_districts} districts')
    print('\n🔍 Showing what was loaded:')
    print(districts_gdf[['district_id', 'district_name']])
    print()
    print('='*80)
    print('⚠️  FALLBACK SYNTHETIC DATA DETECTED!')
    print('='*80)
    print()
    print('💡 FIX:')
    print('1. Check that /content/production_pipeline/data/static/ exists')
    print('2. Check that hdx_boundaries.csv is in that folder')
    print('3. Re-upload the production_pipeline folder')
    print('4. Re-run Step 4')
    print()
    print('⛔ STOPPING — Do NOT proceed with synthetic data!')
    raise ValueError(
        f'Expected 64 real districts, got {num_districts} synthetic districts. '
        f'Historical download would waste hours on fake data.'
    )

# Verify real district names (not fake ones like 'BD-DHA')
fake_indicators = ['BD-DHA', 'BD-KHU', 'BD-RAJ']
district_ids = districts_gdf['district_id'].astype(str).tolist()
found_fake = [fid for fid in fake_indicators if fid in district_ids]
if found_fake:
    print(f'\n❌ DETECTED SYNTHETIC DISTRICT IDs: {found_fake}')
    print('   These are fallback test districts, NOT real Bangladesh districts.')
    raise ValueError(
        f'Synthetic fallback districts detected ({found_fake}). '
        f'Do NOT proceed with historical download.'
    )

# Quick geometry sanity check
bounds = districts_gdf.total_bounds  # [minx, miny, maxx, maxy]
print(f'\n🗺️  Bounding box: lon [{bounds[0]:.2f}, {bounds[2]:.2f}], lat [{bounds[1]:.2f}, {bounds[3]:.2f}]')
if bounds[2] - bounds[0] < 2 or bounds[3] - bounds[1] < 2:
    print('⚠️  WARNING: Bounding box is suspiciously small for all of Bangladesh')
    print('   Expected roughly lon [88, 93], lat [20, 27]')

print(f'\n✅ VERIFICATION PASSED: {num_districts} REAL Bangladesh districts loaded')
print('✅ Safe to proceed with historical download')
print('='*80)


In [ ]:
# STEP 5: Detect latest available dates in GEE
print('='*80)
print('STEP 5 — Detecting data availability')
print('='*80)

try:
    latest_chirps = get_latest_chirps_date()
    print(f'📅 Latest CHIRPS date: {latest_chirps}')
except Exception as e:
    latest_chirps = 'unknown'
    print(f'⚠️ Could not detect CHIRPS latest date: {e}')

try:
    latest_era5 = get_latest_era5_date()
    print(f'📅 Latest ERA5 date  : {latest_era5}')
except Exception as e:
    latest_era5 = 'unknown'
    print(f'⚠️ Could not detect ERA5 latest date: {e}')

# Configuration
START_YEAR = 1981
END_YEAR   = 2023

print(f'\n📋 Download plan:')
print(f'   CHIRPS: {START_YEAR}–{END_YEAR} ({END_YEAR - START_YEAR + 1} years)')
print(f'   ERA5  : {START_YEAR}–{END_YEAR} ({END_YEAR - START_YEAR + 1} years)')
print(f'   Estimated time: 3–4 hours total')
print(f'   Resume: enabled (already-downloaded years will be skipped)')


In [ ]:
# STEP 6: Download CHIRPS 1981–2023 (year by year with checkpointing)
print('='*80)
print(f'STEP 6 — CHIRPS Historical Rainfall ({START_YEAR}–{END_YEAR})')
print('='*80)
print()
print('⏱️  This will take approximately 1.5–2 hours.')
print('    Each year is saved as a checkpoint CSV.')
print('    If interrupted, re-run this cell to resume.')
print()

t0 = time.time()

chirps_raw = extract_chirps_historical(
    districts_gdf,
    start_date=f'{START_YEAR}-01-01',
    end_date=f'{END_YEAR}-12-31',
    output_dir=str(RAW_DIR / 'chirps'),
    resume=True,
)

elapsed = time.time() - t0
print(f'\n{"="*60}')
print(f'✅ CHIRPS download complete!')
print(f'   Rows: {len(chirps_raw):,}')
print(f'   Time: {elapsed/60:.1f} minutes')
print(f'   Districts: {chirps_raw["district_id"].nunique()}')
print(f'   Date range: {chirps_raw["date"].min()} → {chirps_raw["date"].max()}')
print(f'{"="*60}')


In [ ]:
# STEP 7: Compute CHIRPS Climatology
print('='*80)
print('STEP 7 — Computing CHIRPS day-of-year climatology')
print('='*80)

chirps_clim = compute_chirps_climatology(chirps_raw)

chirps_clim_path = HIST_DIR / f'chirps_climatology_{START_YEAR}_{END_YEAR}.csv'
chirps_clim.to_csv(chirps_clim_path, index=False)

print(f'✅ CHIRPS climatology computed and saved')
print(f'   Rows: {len(chirps_clim):,}')
print(f'   File: {chirps_clim_path}')
print(f'   Columns: {list(chirps_clim.columns)}')
print()
print('Sample (first 10 rows):')
chirps_clim.head(10)


In [ ]:
# STEP 8: Download ERA5-Land 1981–2023 (year by year with checkpointing)
print('='*80)
print(f'STEP 8 — ERA5-Land Temperature ({START_YEAR}–{END_YEAR})')
print('='*80)
print()
print('⏱️  This will take approximately 1.5–2 hours.')
print('    Checkpoint/resume is enabled.')
print()

t0 = time.time()

era5_raw = extract_era5_historical(
    districts_gdf,
    start_date=f'{START_YEAR}-01-01',
    end_date=f'{END_YEAR}-12-31',
    variables=['temperature_2m'],
    output_dir=str(RAW_DIR / 'era5'),
    resume=True,
)

elapsed = time.time() - t0
print(f'\n{"="*60}')
print(f'✅ ERA5 download complete!')
print(f'   Rows: {len(era5_raw):,}')
print(f'   Time: {elapsed/60:.1f} minutes')
print(f'   Districts: {era5_raw["district_id"].nunique()}')
print(f'   Date range: {era5_raw["date"].min()} → {era5_raw["date"].max()}')
print(f'{"="*60}')


In [ ]:
# STEP 9: Compute ERA5 Climatology
print('='*80)
print('STEP 9 — Computing ERA5 day-of-year climatology')
print('='*80)

era5_clim = compute_era5_climatology(era5_raw)

era5_clim_path = HIST_DIR / f'era5_climatology_{START_YEAR}_{END_YEAR}.csv'
era5_clim.to_csv(era5_clim_path, index=False)

print(f'✅ ERA5 climatology computed and saved')
print(f'   Rows: {len(era5_clim):,}')
print(f'   File: {era5_clim_path}')
print(f'   Columns: {list(era5_clim.columns)}')
print()
print('Sample (first 10 rows):')
era5_clim.head(10)


In [ ]:
# STEP 10: Save bundled climatology for pipeline reuse
print('='*80)
print('STEP 10 — Save bundled climatology files')
print('='*80)

bundle_dir = save_climatology_bundle(
    chirps_clim=chirps_clim,
    era5_clim=era5_clim,
    output_dir=str(HIST_DIR),
)

print(f'✅ Bundled climatology saved to: {bundle_dir}')


In [ ]:
# STEP 11: Download Summary
print('='*80)
print('DOWNLOAD SUMMARY')
print('='*80)

print()
print('📊 What was downloaded and computed:')
print(f'   CHIRPS raw data   : {len(chirps_raw):>10,} daily district records')
print(f'   ERA5 raw data     : {len(era5_raw):>10,} daily district records')
print(f'   CHIRPS climatology: {len(chirps_clim):>10,} (district × day-of-year)')
print(f'   ERA5 climatology  : {len(era5_clim):>10,} (district × day-of-year)')

print()
print('📁 Files in /content/historical/:')
for f in sorted(HIST_DIR.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'   {f.relative_to(HIST_DIR)}  ({size_mb:.1f} MB)')

print()
print('='*80)
print('📋 NEXT STEPS')
print('='*80)
print()
print('1. Download these files from /content/historical/:')
print(f'   - chirps_climatology_{START_YEAR}_{END_YEAR}.csv')
print(f'   - era5_climatology_{START_YEAR}_{END_YEAR}.csv')
print()
print('2. Place them in your production_pipeline/data/historical/ directory')
print()
print('3. In the main testing notebook (COLAB_STEP_BY_STEP_TEST.ipynb):')
print('   Set HISTORICAL_MODE = "bundled" to use these pre-computed baselines')
print()
print('4. Optionally: also download the raw/ folder as backup')
print('   (allows recomputing climatology without re-downloading from GEE)')
print()
print('✅ Historical baseline download is COMPLETE!')
